## Observações

Importações identificadas como possivelmente não utilizadas:

- `from optbinning import OptimalBinning`
- `from sklearn.linear_model import LogisticRegression`
- `from sklearn.metrics import classification_report`
- `from sklearn.metrics import confusion_matrix`
- `from sklearn.model_selection import RandomizedSearchCV`
- `import lightgbm as lgb`
- `import matplotlib.pyplot as plt`
- `import pickle`
- `import seaborn as sns`
- `import shap`

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [4]:
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import shap
import pickle
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

from optbinning import OptimalBinning

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [5]:
import optuna
import xgboost as xgb
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score
)

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [6]:
X_train = pd.read_parquet("../data/processed/X_train.parquet")
y_train = pd.read_parquet("../data/processed/y_train.parquet")

X_test = pd.read_parquet("../data/processed/X_test.parquet")
y_test = pd.read_parquet("../data/processed/y_test.parquet")

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [7]:
cols_woes = []
cols_norm = []
for col in X_train.columns:
    if "_woe" in col:
        cols_woes.append(col)
    else:
        cols_norm.append(col)

## Model 

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [8]:
def optimize_xgb(X, y, n_trials=30):

    def objective(trial):

        params = {
            "objective": "binary:logistic",
            "eval_metric": "auc",

            "n_estimators": trial.suggest_int(
                "n_estimators",
                100,
                1000
            ),

            "learning_rate": trial.suggest_float(
                "learning_rate",
                0.01,
                0.1,
                log=True
            ),

            "max_depth": trial.suggest_int(
                "max_depth",
                3,
                10
            ),

            "min_child_weight": trial.suggest_int(
                "min_child_weight",
                1,
                20
            ),

            "subsample": trial.suggest_float(
                "subsample",
                0.7,
                1.0
            ),

            "colsample_bytree": trial.suggest_float(
                "colsample_bytree",
                0.7,
                1.0
            ),

            "gamma": trial.suggest_float(
                "gamma",
                0,
                5
            ),

            "reg_alpha": trial.suggest_float(
                "reg_alpha",
                0,
                5
            ),

            "reg_lambda": trial.suggest_float(
                "reg_lambda",
                0,
                5
            ),

            "random_state": 42,
            "n_jobs": -1
        }

        model = xgb.XGBClassifier(**params)

        cv = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=42
        )

        auc = cross_val_score(
            model,
            X,
            y,
            scoring="roc_auc",
            cv=cv,
            n_jobs=-1
        ).mean()

        return auc

    study = optuna.create_study(
        direction="maximize"
    )

    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=True
    )

    return study.best_params

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [9]:
best_params_norm = optimize_xgb(
    X_train[cols_norm],
    y_train,
    n_trials=30
)

xgb_norm = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    **best_params_norm
)

xgb_norm.fit(
    X_train[cols_norm],
    y_train
)

[I 2026-06-07 20:58:28,197] A new study created in memory with name: no-name-de104259-15d1-4bb3-b945-af6580e20c8c
Best trial: 0. Best value: 0.86496:   3%|▎         | 1/30 [00:05<02:47,  5.79s/it]

[I 2026-06-07 20:58:33,986] Trial 0 finished with value: 0.86495954361506 and parameters: {'n_estimators': 568, 'learning_rate': 0.038435715672140054, 'max_depth': 4, 'min_child_weight': 15, 'subsample': 0.9834521989354317, 'colsample_bytree': 0.7667638295714473, 'gamma': 2.525463471759613, 'reg_alpha': 3.627799197079834, 'reg_lambda': 3.128106058267528}. Best is trial 0 with value: 0.86495954361506.


Best trial: 0. Best value: 0.86496:   7%|▋         | 2/30 [00:12<02:53,  6.21s/it]

[I 2026-06-07 20:58:40,492] Trial 1 finished with value: 0.8646328966811755 and parameters: {'n_estimators': 552, 'learning_rate': 0.02956395687006422, 'max_depth': 5, 'min_child_weight': 11, 'subsample': 0.9298822796708677, 'colsample_bytree': 0.8526415890493221, 'gamma': 0.7830262277619643, 'reg_alpha': 2.4150833915085963, 'reg_lambda': 1.1173712030807885}. Best is trial 0 with value: 0.86495954361506.


Best trial: 2. Best value: 0.865184:  10%|█         | 3/30 [00:17<02:34,  5.71s/it]

...

Best trial: 27. Best value: 0.865557:  97%|█████████▋| 29/30 [02:31<00:05,  5.15s/it]

[I 2026-06-07 21:01:00,019] Trial 28 finished with value: 0.8654263919138444 and parameters: {'n_estimators': 646, 'learning_rate': 0.035240678726085946, 'max_depth': 10, 'min_child_weight': 19, 'subsample': 0.8218130660327359, 'colsample_bytree': 0.8301482163117055, 'gamma': 2.875585179973148, 'reg_alpha': 3.085146631609971, 'reg_lambda': 4.683344263232884}. Best is trial 27 with value: 0.8655570710511034.


Best trial: 27. Best value: 0.865557: 100%|██████████| 30/30 [02:35<00:00,  5.18s/it]


[I 2026-06-07 21:01:03,691] Trial 29 finished with value: 0.8647686038403872 and parameters: {'n_estimators': 479, 'learning_rate': 0.035734382745410416, 'max_depth': 10, 'min_child_weight': 14, 'subsample': 0.8813817644904788, 'colsample_bytree': 0.9088437172888073, 'gamma': 2.35905800796434, 'reg_alpha': 2.614599056951577, 'reg_lambda': 3.4511938159734745}. Best is trial 27 with value: 0.8655570710511034.


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8297175671601488
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import 

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [10]:
best_params_woe = optimize_xgb(
    X_train[cols_woes],
    y_train,
    n_trials=30
)

xgb_woe = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    **best_params_woe
)

xgb_woe.fit(
    X_train[cols_woes],
    y_train
)

[I 2026-06-07 21:01:04,903] A new study created in memory with name: no-name-4c28b211-6f8c-477e-a3c5-c7fd6a24be83
Best trial: 0. Best value: 0.862668:   3%|▎         | 1/30 [00:05<02:36,  5.40s/it]

[I 2026-06-07 21:01:10,305] Trial 0 finished with value: 0.8626684420446313 and parameters: {'n_estimators': 629, 'learning_rate': 0.018460094690036583, 'max_depth': 10, 'min_child_weight': 13, 'subsample': 0.8308163417617744, 'colsample_bytree': 0.9571696462942321, 'gamma': 0.8238386030786038, 'reg_alpha': 4.71612652832149, 'reg_lambda': 1.774831016653402}. Best is trial 0 with value: 0.8626684420446313.


Best trial: 1. Best value: 0.863518:   7%|▋         | 2/30 [00:10<02:24,  5.15s/it]

[I 2026-06-07 21:01:15,277] Trial 1 finished with value: 0.8635183785383834 and parameters: {'n_estimators': 844, 'learning_rate': 0.07621583495551201, 'max_depth': 3, 'min_child_weight': 7, 'subsample': 0.9223037103679983, 'colsample_bytree': 0.7863543028941392, 'gamma': 1.6341722970844015, 'reg_alpha': 0.8768253643507834, 'reg_lambda': 4.860995376688227}. Best is trial 1 with value: 0.8635183785383834.


Best trial: 2. Best value: 0.863635:  10%|█         | 3/30 [00:13<01:59,  4.44s/it]

...

Best trial: 26. Best value: 0.86389:  97%|█████████▋| 29/30 [02:05<00:05,  5.05s/it]

[I 2026-06-07 21:03:10,710] Trial 28 finished with value: 0.8638613574623808 and parameters: {'n_estimators': 686, 'learning_rate': 0.045331739469282414, 'max_depth': 3, 'min_child_weight': 15, 'subsample': 0.8601205097456672, 'colsample_bytree': 0.729498445971838, 'gamma': 1.107392377988565, 'reg_alpha': 2.7626642228418867, 'reg_lambda': 3.9087958915286456}. Best is trial 26 with value: 0.8638902718884577.


Best trial: 26. Best value: 0.86389: 100%|██████████| 30/30 [02:10<00:00,  4.34s/it]


[I 2026-06-07 21:03:15,205] Trial 29 finished with value: 0.8637112871145118 and parameters: {'n_estimators': 668, 'learning_rate': 0.04471345689439287, 'max_depth': 5, 'min_child_weight': 14, 'subsample': 0.8560597304573037, 'colsample_bytree': 0.7061696172159745, 'gamma': 1.141399166800465, 'reg_alpha': 2.7951145158470148, 'reg_lambda': 4.421633850490902}. Best is trial 26 with value: 0.8638902718884577.


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.7142758916378572
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import 

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [ ]:
def evaluate_model(
    model,
    X_test,
    y_test,
    model_name
):

    y_prob = model.predict_proba(X_test)[:, 1]

    y_pred = model.predict(X_test)

    auc = roc_auc_score(
        y_test,
        y_prob
    )

    gini = 2 * auc - 1

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    return {
        "model": model_name,
        "auc": round(auc, 4),
        "gini": round(gini, 4),
        "accuracy": round(accuracy, 4)
        
    }

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [12]:
result_norm = evaluate_model(
    xgb_norm,
    X_test[cols_norm],
    y_test,
    "XGB_NORM"
)

result_woe = evaluate_model(
    xgb_woe,
    X_test[cols_woes],
    y_test,
    "XGB_WOE"
)

results = pd.DataFrame([
    result_norm,
    result_woe
])



### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [14]:
results

,model,auc,gini,accuracy
0,XGB_NORM,0.8668,0.7335,0.9372
1,XGB_WOE,0.8636,0.7273,0.9365
